In [157]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import requests
from IPython.display import Markdown, display, clear_output
import gradio as gr
import json
from agents import Agent, Runner, trace, function_tool, ModelSettings
from langchain_community.utilities import GoogleSerperAPIWrapper
import ipywidgets as widgets
import re

In [158]:
load_dotenv(override=True)
# openai=OpenAI()

True

In [159]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
os.getenv('GOOGLE_API_KEY')

openai = OpenAI(api_key=openai_api_key)

In [160]:
# OLLAMA_BASE_URL = "http://localhost:11434/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

In [161]:
# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
# gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)

In [162]:
serper = GoogleSerperAPIWrapper()
serper_images = GoogleSerperAPIWrapper(type="images")

In [163]:
 
request = f"""
You are a content agent writing on the recent trends in the Technology. Your job:
 
Write a ~500-word blog post on: "AI Agents"
   - Tone: practical, friendly, no fluff
   - Include a short intro, 5 numbered sections with subheads, and a 2-sentence conclusion
   - Target audience: engineers
 
"""
messages = [{"role": "user", "content": request}]

In [164]:
messages

[{'role': 'user',
  'content': '\nYou are a content agent writing on the recent trends in the Technology. Your job:\n\nWrite a ~500-word blog post on: "AI Agents"\n   - Tone: practical, friendly, no fluff\n   - Include a short intro, 5 numbered sections with subheads, and a 2-sentence conclusion\n   - Target audience: engineers\n\n'}]

In [165]:
prompt = """
You are a content agent responsible for producing and publishing content 
for our platform. Given a category and subtopic, follow this process 
IN ORDER, using the tools available to you:

1. RESEARCH
   Call web_search tool with a short, specific query (4-6 words) to gather 
   current, factual context on the subtopic within the category. This is 
   to avoid generic or outdated filler — do not skip this step.
   You may call it up to 2 times if the first results are too broad or irrelevant.

2. WRITE CONTENT
   Using the research, write a piece with:
   - title (SEO-friendly, max 70 characters)
   - intro (2-3 sentences)
   - sections (3-5, each with a heading and body text)
   - conclusion (2-3 sentences)
   - tags (3-5 relevant SEO tags)
   Do not fabricate facts, statistics, or quotes not supported by the research.
   Word count target: word_count words total.

3. SOURCE IMAGES
   Call image_search tool once per needed image (3-5 total):
   - 1 hero/featured image — landscape orientation
   - 2-4 supporting images, one per relevant section
   Only use royalty-free sources. Return the image URL and source name for each.
   If no suitable image is found for a section, skip it rather than 
   inventing a URL.

4. ASSEMBLE DRAFT
   Combine the written content and images into a single JSON object 
   matching this structure:
   {
     "title": "", "slug": "", "category": "", "tags": [],
     "meta_description": "", "intro": "",
     "sections": [{"heading": "", "text": "", "image": {"url": "", "source": ""}}],
     "conclusion": "", "featured_image": {"url": "", "source": ""},
     "status": "draft"
   }

5. STOP FOR HUMAN APPROVAL
   Do NOT call publish tool yet. Present the assembled draft as your final 
   response for this turn, clearly labeled, and wait for explicit approval 
   before publishing.

6. PUBLISH (only after approval is given )
   Call publish tool with the approved payload. Set "status" to "draft" or 
   "live" based on what the human specifies.

7. REPORT
   After publish tool returns, report back the URL/ID and a 1-line summary.

Rules:
- Follow the steps in order — do not skip research or jump straight to writing.
- Never call publish tool without explicit human approval in the conversation.
- If any tool call fails, report the error and stop — do not retry more than once.
- Do not fabricate image URLs, facts, or statistics.
"""

In [166]:
def web_search(query: str)->str:
    """ Search the web for the current information on a given query"""
    return serper.run(query)

In [167]:
web_search_json = {
    "name": "web_search",
    "description": "Search the web for current information relevant to the topic",
    "parameters":{ 
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "A short, specific search query to seach the web(4-6 words)"
            } 
        },
    "required": ["query"],
    "additionalProperties": False
    }
}

In [168]:
def image_search(query):
    """Search for royalty-free images relevant to the query."""
    results = serper_images.results(query)  # use .results(), not .run(), to get structured data

    images = results.get("images", [])[:5]  # top 5 image results
    if not images:
        return []

    return [
        {"url": img.get("imageUrl"), "source": img.get("source", "Unknown")}
        for img in images
    ]

In [169]:
image_search_json={
    "name": "image_search",
    "description": "Search the web for royalty free images on the platforms like unsplash, pixabay,p exels, etc relevant to the query",
    "parameters":{
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "A short, specific search query to search for images(4-6 words)"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [170]:
def publish(payload):
    """Publish the content draft to the platform."""
    return payload

In [171]:
publish_json={
    "name": "publish",
    "description": "Publish the finished content draft to the platform.",
    "parameters":{
        "type": "object",
        "properties":{
            "payload":{
                "type": "string",
                "description": "The final content draft + images to publish"
            }
        },
        "required": ["payload"],
        "additionalProperties": False
    }
}

In [172]:
tools = [{"type": "function", "function": web_search_json}, {"type": "function", "function": image_search_json}, {"type": "function", "function": publish_json}]

In [173]:
tools_flow = {
    "web_search": web_search,
    "image_search": image_search,
    "publish": publish,
}                         

In [174]:
def agent01(category, subtopic, word_count):
    messages = [
        {"role": "system", "content": prompt}, 
        {"role": "user", "content": f"category: {category}, subtopic: {subtopic}"}
    ]
    response = openai.chat.completions.create(model = "gpt-5.4-mini", messages = messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_flow[function_name](**args)
            messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model = "gpt-5.4-mini", messages = messages, tools=tools)

    return response.choices[0].message.content, messages

In [175]:
def review_draft(messages, decision, feedback=None, live=False):
    """
    decision: "approve" or "reject"
    feedback: required if decision == "reject" — what to fix
    live: only used if decision == "approve" — True = publish live, False = save as draft
    """
    if decision == "reject":
        if not feedback:
            raise ValueError("Feedback is required when rejecting a draft.")
        user_msg = f"Not approved. Please revise the draft based on this feedback: {feedback}"
    elif decision == "approve":
        user_msg = f"Approved. Publish with status = {'live' if live else 'draft'}."
    else:
        raise ValueError("decision must be 'approve' or 'reject'")

    messages.append({"role": "user", "content": user_msg})
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_flow[fn_name](**args)
            messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    return response.choices[0].message.content, messages

In [176]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def show_approval_buttons(messages):
    output = widgets.Output()

    approve_btn = widgets.Button(description="Approve (Draft)", button_style="success")
    approve_live_btn = widgets.Button(description="Approve (Live)", button_style="info")
    reject_btn = widgets.Button(description="Reject", button_style="danger")

    def on_approve(b, live=False):
        with output:
            clear_output()
            print("Publishing..." if live else "Saving as draft...")
            result = review_draft(messages, live=live)
            print(result)

    def on_reject(b):
        with output:
            clear_output()
            print("Rejected. Add feedback and re-run agent01 with revised instructions.")

    approve_btn.on_click(lambda b: on_approve(b, live=False))
    approve_live_btn.on_click(lambda b: on_approve(b, live=True))
    reject_btn.on_click(on_reject)

    display(widgets.HBox([approve_btn, approve_live_btn, reject_btn]), output)

In [177]:
def clean_json_string(s):
    s = s.strip()

    # strip markdown code fences if present
    if s.startswith("```"):
        s = s.split("\n", 1)[1]
        s = s.rsplit("```", 1)[0]
        s = s.strip()

    # extract only the first complete {...} block, ignore any trailing text
    match = re.search(r'\{.*\}', s, re.DOTALL)
    if match:
        s = match.group(0)

    return s.strip()

In [178]:

def display_draft(draft_json_str):
    draft_json_str = clean_json_string(draft_json_str)
    data = json.loads(draft_json_str)

    md = f"# {data.get('title', '(no title)')}\n\n"
    md += f"*{data.get('meta_description', '')}*\n\n"
    md += f"**Category:** {data.get('category', '')} | **Tags:** {', '.join(data.get('tags', []))}\n\n"
    md += f"---\n\n{data.get('intro', '')}\n\n"

    for section in data.get('sections', []):
        md += f"## {section.get('heading', '')}\n\n{section.get('text', '')}\n\n"
        image = section.get('image', {})
        if image.get('url'):
            md += f"![{section.get('heading','')}]({image['url']})\n\n"
        else:
            md += "*(no image sourced for this section)*\n\n"

    md += f"---\n\n{data.get('conclusion', '')}\n\n"
    featured = data.get('featured_image', {})
    if featured.get('url'):
        md += f"**Featured image:** {featured['url']}\n\n"
    md += f"**Status:** {data.get('status', 'unknown')}"

    display(Markdown(md))

In [ ]:
draft, messages = agent01(category="Technology", subtopic="AI Agents in 2026", word_count=600)
display_draft(draft)

# print(repr(draft))


# AI Agents in 2026: Trends, Use Cases, and Governance

*Explore how AI agents are shifting from chat to action in 2026, with trends in enterprise adoption, orchestration, and governance.*

**Category:** Technology | **Tags:** AI Agents, Agentic AI, Enterprise AI, AI Governance, Automation

---

AI agents are moving from experimental assistants to operational systems that can take action across workflows. In 2026, the big story is not just smarter models, but how organizations govern, orchestrate, and validate agentic automation.

## From chatbots to action-oriented agents

The clearest shift in 2026 is that AI is moving beyond conversation and into execution. Instead of answering questions only, agents are increasingly expected to carry out tasks, manage context, and interact with tools inside real business workflows.

![From chatbots to action-oriented agents](https://www.shutterstock.com/image-photo/businessman-interacting-ai-agent-interface-260nw-2713504885.jpg)

## Enterprise adoption is becoming practical

Current reporting points to organizations focusing less on hype and more on validation. Teams are looking for near-term impact, clearer use cases, and agent systems that can support productivity across functions like customer service, development, and internal operations.

![Enterprise adoption is becoming practical](https://framerusercontent.com/images/KlIOEQhJaCEzUxeVqVRhXjRAo9c.png?width=5792&height=3600)

## Orchestration and multi-agent workflows matter more

As agent use expands, orchestration becomes a core design issue. Research highlights multi-agent systems, cross-functional adoption, and workflow coordination as important themes for 2026, because isolated agents are less useful than systems that can hand off tasks reliably.

![Orchestration and multi-agent workflows matter more](https://cdn.prod.website-files.com/617892070bc7e35f30380d8e/68ceca1f11019eb9dcab70d4_imgi_1_AI_agents_dashboard_interface_e2c753ea-oaiwS2Df.webp)

## Governance and security are now central

The more autonomy agents get, the more important governance becomes. Enterprise reporting for 2026 emphasizes the need for oversight, guardrails, and security controls so organizations can manage risk while still benefiting from automation.

![Governance and security are now central](https://www.paloaltonetworks.com/content/dam/pan/en_US/images/cyberpedia/what-is-agentic-ai-governance/Agentic-AI-governance-across-the-lifecycle.png?imwidth=480)

---

AI agents in 2026 are best understood as a shift from generating text to completing work. The winners will likely be the organizations that pair useful automation with strong orchestration, governance, and realistic rollout plans.

**Featured image:** https://media.deloitte.com/is/image/deloitte/agents-hub-1920x360:Mobile?$Responsive$&fmt=webp&fit=stretch,1&dpr=on,2.625

**Status:** draft

In [ ]:
revised_draft, messages = review_draft(messages, decision="reject", feedback="Make the tone more casual and add a section on AI agent security risks.")
print(repr(revised_draft))
display_draft(revised_draft) 
show_approval_buttons(messages)

'Draft assembled and ready for approval:\n\n```json\n{\n  "title": "AI Agents in 2026: What’s Changing and What to Watch",\n  "slug": "ai-agents-in-2026-whats-changing-and-what-to-watch",\n  "category": "Technology",\n  "tags": [\n    "AI agents",\n    "agentic AI",\n    "enterprise AI",\n    "AI security",\n    "automation"\n  ],\n  "meta_description": "A casual, practical look at AI agents in 2026, from real-world use cases to the security risks teams need to plan for.",\n  "intro": "AI agents are moving fast in 2026, and the big shift is pretty simple: they’re doing more than chatting. Instead of just answering questions, they’re starting to take actions, handle workflows, and work across tools with more autonomy. That makes them useful — but it also means teams need to think about control, security, and real-world limits.",\n  "sections": [\n    {\n      "heading": "1. AI agents are moving from chat to action",\n      "text": "The headline for 2026 is that AI agents are becoming mo

# AI Agents in 2026: What’s Changing and What to Watch

*A casual, practical look at AI agents in 2026, from real-world use cases to the security risks teams need to plan for.*

**Category:** Technology | **Tags:** AI agents, agentic AI, enterprise AI, AI security, automation

---

AI agents are moving fast in 2026, and the big shift is pretty simple: they’re doing more than chatting. Instead of just answering questions, they’re starting to take actions, handle workflows, and work across tools with more autonomy. That makes them useful — but it also means teams need to think about control, security, and real-world limits.

## 1. AI agents are moving from chat to action

The headline for 2026 is that AI agents are becoming more useful in everyday work. The trend is less about flashy demos and more about getting actual tasks done: helping developers in the command line, supporting customer service, and automating multi-step workflows. In other words, agents are starting to look less like novelty assistants and more like practical coworkers.

![1. AI agents are moving from chat to action](https://media.deloitte.com/is/image/deloitte/agents-hub-1920x360:Mobile?$Responsive$&fmt=webp&fit=stretch,1&dpr=on,2.625)

## 2. Businesses want proof, not hype

A lot of organizations are shifting from experimentation to validation. They want to know what actually works before they scale AI agents across the business. That means looking closely at measurable impact, workflow fit, and whether an agent really saves time instead of adding another layer of complexity. The vibe in 2026 is basically: show me the value.

![2. Businesses want proof, not hype](https://cdn.prod.website-files.com/617892070bc7e35f30380d8e/68ceca1f11019eb9dcab70d4_imgi_1_AI_agents_dashboard_interface_e2c753ea-oaiwS2Df.webp)

## 3. Multi-agent systems are getting more attention

Another big 2026 theme is coordination. Instead of one agent trying to do everything, teams are exploring setups where multiple agents each handle a piece of the job. That can be powerful for cross-functional work, but it also raises the bar for orchestration, monitoring, and making sure the pieces don’t step on each other.

![3. Multi-agent systems are getting more attention](https://framerusercontent.com/images/KlIOEQhJaCEzUxeVqVRhXjRAo9c.png?width=5792&height=3600)

## 4. AI agent security risks are a real deal

The security story matters a lot more once agents can take actions. Research points to risks like prompt injection, tool manipulation, excessive permissions, data exposure, and theft of agent credentials. Because agents can connect to real systems and act quickly, a small mistake can turn into a bigger problem fast. The practical takeaway: keep permissions tight, watch what tools agents can reach, and don’t treat autonomy as the same thing as trust.

![4. AI agent security risks are a real deal](https://www.xenonstack.com/hs-fs/hubfs/ai-agent-vulnerabilities.png?width=1920&height=1080&name=ai-agent-vulnerabilities.png)

## 5. The best teams will balance speed with guardrails

The most successful AI agent setups in 2026 are likely to be the ones that combine useful automation with clear boundaries. That means human review where it matters, governance that people actually follow, and simple rules around data access and escalation. The goal isn’t to slow everything down — it’s to make sure the gains from agents are worth the risk.

![5. The best teams will balance speed with guardrails](https://towardsdatascience.com/wp-content/uploads/2024/07/11zQxaR7OuAKdxmm1Z_c1kg-1.png)

---

AI agents in 2026 are shaping up to be genuinely useful, not just interesting. But the teams that get the most out of them will be the ones that stay practical: focus on real workflows, keep the tone of deployment calm and measured, and build security in from the start. In short, agents can do more now — so your guardrails need to do more too.

**Featured image:** https://media.deloitte.com/is/image/deloitte/agents-hub-1920x360:Mobile?$Responsive$&fmt=webp&fit=stretch,1&dpr=on,2.625

**Status:** draft

Output()

In [ ]:
result, messages = review_draft(messages, decision="approve", live=False)
print(result)

{"title":"AI Agents in 2026: Trends, Use Cases, and What to Expect","slug":"ai-agents-in-2026","category":"Technology","tags":["AI agents","agentic AI","AI automation","enterprise AI","AI workflows"],"meta_description":"A practical look at how AI agents are evolving in 2026, from software development to enterprise workflows and customer service.","intro":"AI agents are moving beyond simple chat interfaces and into real work execution. In 2026, the biggest shift is less about novelty and more about reliability, integration, and measurable business value.\n\nAs organizations adopt agentic AI, the focus is shifting toward workflows, oversight, and specific tasks that can be delegated safely.","sections":[{"heading":"1. AI agents are moving from chat to action","text":"Research around 2026 AI trends points to a clear change: agents are expected to do more than answer questions. They are being positioned to complete tasks, connect to tools, and carry context across steps, which makes them more useful in real workflows than earlier chatbot-style systems.","image":{"url":"","source":""}},{"heading":"2. Developers are adopting CLI-first AI agents","text":"One current trend highlighted in the research is the rise of command-line and terminal-based AI agents. These tools are becoming important in software development because they fit directly into coding, debugging, refactoring, and testing workflows where developers already spend time.","image":{"url":"","source":""}},{"heading":"3. Enterprises want validated automation","text":"The research also suggests that organizations are shifting from experimentation to validation. Instead of asking whether AI agents are interesting, teams are asking which workflows they can trust them to handle, how they integrate with existing systems, and how to measure success.","image":{"url":"","source":""}},{"heading":"4. Customer service and operations are prime use cases","text":"Another theme in the research is the move toward more proactive customer service and broader operational support. AI agents are being discussed as tools that can help manage routine requests, surface information, and reduce manual handoffs across business functions.","image":{"url":"","source":""}}],"conclusion":"AI agents in 2026 are best understood as workflow tools rather than novelty demos. The organizations that benefit most will be the ones that start with high-value tasks, add oversight, and expand only after proving reliability.\n\nAs adoption grows, the winners will likely be those that combine automation with strong human control and clear business goals.","featured_image":{"url":"","source":""},"status":"draft"}

In [ ]:
# !ollama pull llama3.2
 
# model_name = "llama3.2"
# model_name = "gpt-5.4-mini"
# model_name = "gemini-3.1-flash-lite"
 

In [ ]:
# response = ollama.chat.completions.create(model=model_name, messages=messages)
# response = openai.chat.completions.create(model=model_name, messages=messages)
# response = gemini.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# print(answer)

In [ ]:
display(Markdown(answer))

NameError: name 'answer' is not defined